In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

In [ ]:
df_epi_raw = pd.read_csv("data/epi_r.csv", sep=",")
df_epi_raw.info()

## Duplicates


In [ ]:
df_epi_raw.duplicated().sum()

In [ ]:
df_epi_drop_dup = df_epi_raw.drop_duplicates()

## DownCast 

In [ ]:
(df_epi_drop_dup.nunique()).value_counts()

Т.е. 674 столбцы имеют 2 униткальных значения, 1 столбец - 17736, 1 - 8 уникальных значений.

Найдем столбцы с 2 уникальными значениями.

In [ ]:
mask_cols_eq2 = df_epi_drop_dup.nunique() == 2
cols_eq2 = df_epi_drop_dup.columns[mask_cols_eq2]

In [ ]:
df_epi_drop_dup.columns[~mask_cols_eq2]

In [ ]:
df_epi_drop_dup.info(memory_usage=True)

In [ ]:
df_epi = df_epi_drop_dup.copy()

In [ ]:
for column in df_epi_drop_dup.columns[mask_cols_eq2]:
    df_epi[column] = pd.to_numeric(df_epi[column], downcast="integer")

In [ ]:
df_epi.info(memory_usage=True)

## Columns filter

Check:
    - frequencies (kiwi not pass)
    - correlation with other columns
    - parcers (ingredient_slicer, pyfood)

In [ ]:
import pandas as pd
import re
from pipe import StrictIngredientSelector

all_columns_eq2 = list(df_epi.columns[mask_cols_eq2])

selector = StrictIngredientSelector()
final_ingredients = selector.filter_columns(all_columns_eq2)

print(f"Осталось чистых ингредиентов: {len(final_ingredients)}")
print("Первые элементы списка:", final_ingredients[:15])

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from pipe import DataCleaner, FeatureExtractor


from sklearn.pipeline import Pipeline

preprocessing_pipe = Pipeline(
    [("data_cleaner", DataCleaner()), ("feature_extractor", FeatureExtractor())]
)

preprocessing_pipe.fit_transform(df_epi_raw)

In [ ]:
pd.Series(list(set(all_columns_eq2) - set(final_ingredients))).to_csv(
    "data/not_ingredient_cols.csv"
)

После ручного просмотра оставшихся не ингредиентов были выявлены следующие - 'marshmallow', 'créme de cacao', 'yogurt', 'bulgur', 'kirsch', 'swordfish', 'chambord', 'collard greensa'

In [ ]:
pd.Series(list(set(final_ingredients))).to_csv("data/ingredient_cols.csv")

In [ ]:
def titles_by_ingredients(col, df):
    return df[df[col] == 1][["title", "rating"]]


titles_by_ingredients("spirit", df_epi)

In [ ]:
assert 1 == 0